# Exp7.2.6.2 — Mean-CE output-bias ablation

Analysis-only notebook. It reads finalized aggregate artifacts and does not train or run inference.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=None):
    path = (start or Path.cwd()).resolve()
    for candidate in (path, *path.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('Could not find repository root')

REPO_ROOT = find_repo_root()
root = REPO_ROOT / 'notebooks' / 'artifacts' / 'experiment_7_2_6_2_mean_ce_bias' / 'mean_ce_bias_v1'
native = pd.read_csv(root / 'native_performance_summary.csv')
decomp = pd.read_csv(root / 'bias_effect_decomposition_summary.csv')
probes = pd.read_csv(root / 'l2_probe_summary.csv')
probe_delta = pd.read_csv(root / 'l2_probe_delta_summary.csv')
bias_diag = pd.read_csv(root / 'bias_diagnostics_summary.csv')
bias_values = pd.read_csv(root / 'bias_values.csv')
history = pd.read_csv(root / 'training_history_summary.csv')

## 1. Native BA: no-bias vs bias-trained

In [ ]:
display(native[native['split'] == 'test'].sort_values('condition'))

## 2. Exact bias-effect decomposition

Total effect = direct readout effect + E2E training effect.

In [ ]:
display(decomp.sort_values('split'))

## 3. L2 representation accessibility

In [ ]:
display(probes[probes['split'] == 'test'].sort_values(['probe', 'condition']))
display(probe_delta.sort_values('probe'))

## 4. Learned bias diagnostics

In [ ]:
display(bias_diag.sort_values('split'))
display(bias_values.sort_values(['class_label', 'seed']))

## 5. Aggregate validation curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for condition, frame in history.groupby('condition'):
    if 'val_ba_mean' in frame.columns:
        ax.plot(frame['epoch'], frame['val_ba_mean'], label=condition)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation balanced accuracy')
ax.set_ylim(0, 1)
ax.legend()
ax.set_title('Mean-CE bias ablation')
plt.show()